# 04 Feature Baseline EDA

Purpose:
- analyze extracted handcrafted features
- compare class separability and feature redundancy
- prepare the ground for the first baseline model

Primary questions:
- Which features show separation between classes?
- Which features are redundant or unstable?
- How different do file-level and patient-level views look?
- Which modeling assumptions need to be revisited before training?


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Fixed features location under backend/data
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'backend' / 'data'
FEATURES_PATH = DATA_DIR / 'processed' / 'features.csv'
print('Looking for features at', FEATURES_PATH)
FEATURES_PATH

In [ ]:
# Feature baseline EDA
from pathlib import Path
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.decomposition import PCA
except Exception as e:
    print('Some packages are missing for feature EDA:', e)
    pd = None

FEATURES_PATH = FEATURES_PATH if 'FEATURES_PATH' in globals() else (Path.cwd().resolve() / 'backend' / 'data' / 'processed' / 'features.csv')
print('Looking for features at', FEATURES_PATH)
if not FEATURES_PATH.exists():
    print('Features CSV not found at expected location; please run feature extraction or place a features.csv file under backend/data/processed')
else:
    if pd is None:
        print('pandas not available; install backend[dev] to run this cell')
    else:
        df = pd.read_csv(FEATURES_PATH)
        display(df.head())
        print('Shape:', df.shape)
        # basic stats
        display(df.describe())
        # find numeric features and optional label column
        num = df.select_dtypes(include=[np.number])
        print('Numeric feature count:', num.shape[1])
        # correlation heatmap for first 30 numeric features to keep plot readable
        corr = num.corr()
        if 'sns' in globals():
            plt.figure(figsize=(10, 8))
            sns.heatmap(corr.iloc[:30, :30], cmap='RdBu_r', center=0)
            plt.title('Feature correlation (first 30 features)')
            plt.show()
        # PCA scatter if label exists
        label_candidates = [c for c in df.columns if c.lower() in ('label','diagnosis','target','class','phq9','phq')]
        labels = df[label_candidates[0]] if label_candidates else None
        pca = PCA(n_components=2)
        Z = pca.fit_transform(num.fillna(0))
        if 'plt' in globals():
            plt.figure(figsize=(8, 6))
            if labels is not None:
                for lab in sorted(pd.unique(labels)):
                    mask = labels == lab
                    plt.scatter(Z[mask, 0], Z[mask, 1], label=str(lab), alpha=0.6)
                plt.legend()
            else:
                plt.scatter(Z[:, 0], Z[:, 1], alpha=0.6)
            plt.title('PCA (2 components) of numeric features')
            plt.xlabel('PC1')
            plt.ylabel('PC2')
            plt.show()